# Ensemble Stacking for Car Damage Detection (YOLO Models)
# ===========================================
# 
# This notebook implements ensemble stacking using multiple YOLO detection models
# (YOLOv8, YOLOv11, YOLOv12) to improve object detection performance on car damage detection.
#
# **Stacking Strategy:**
# - Base Models: YOLOv8, YOLOv11, YOLOv12
# - Meta-Learner: Weighted NMS / Regression for boxes / Classification for classes
# - Cross-Validation: K-Fold for out-of-fold predictions
# - Ensemble Methods: Weighted NMS, Box Coordinate Regression, Class Probability Stacking


# 1. Imports and Setup


In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# YOLO/Ultralytics
from ultralytics import YOLO
import torch

# Computer Vision
import cv2
from PIL import Image

# Scikit-learn
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder

# XGBoost (if available)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Using Logistic Regression as meta-learner.")

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# Utilities
from collections import defaultdict
import json
from pathlib import Path

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


# 2. Configuration and Paths


In [ ]:
# Dataset paths (Kaggle format)
TRAIN_DIR = '/kaggle/input/car-damage-detection/Car_Demage_Detection_Datasets/CarDD_release/CarDD_COCO/train'
VAL_DIR = '/kaggle/input/car-damage-detection/Car_Demage_Detection_Datasets/CarDD_release/CarDD_COCO/val'
ANNOTATIONS_DIR = '/kaggle/input/car-damage-detection/Car_Demage_Detection_Datasets/CarDD_release/CarDD_COCO/annotations'

# Model weights directory
MODELS_DIR = '/kaggle/input/car-damage-models'  # Adjust based on your dataset structure
LOCAL_MODELS_DIR = '../Models_Weight_files'

# Model paths
MODEL_PATHS = {
    'yolov8': os.path.join(LOCAL_MODELS_DIR, 'bestyolov8.pt') if os.path.exists(LOCAL_MODELS_DIR) else None,
    'yolov11': os.path.join(LOCAL_MODELS_DIR, 'best(1)yolov11.pt') if os.path.exists(LOCAL_MODELS_DIR) else None,
    'yolov12': os.path.join(LOCAL_MODELS_DIR, 'best(1)yolov12.pt') if os.path.exists(LOCAL_MODELS_DIR) else None,
}

# Parameters
IMG_SIZE = 640  # YOLO input size
CONF_THRESHOLD = 0.25  # Confidence threshold for detections
IOU_THRESHOLD = 0.45  # IoU threshold for NMS
N_SPLITS = 5  # Number of folds for cross-validation

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


# 3. Helper Functions for YOLO Detection


In [ ]:
def load_yolo_model(model_name, model_path=None):
    """
    Load YOLO model
    
    Args:
        model_name: 'yolov8', 'yolov11', or 'yolov12'
        model_path: Path to model weights file
    """
    if model_path and os.path.exists(model_path):
        print(f"Loading {model_name} from {model_path}")
        return YOLO(model_path)
    else:
        # Load pre-trained model (will download if not available)
        print(f"Loading pre-trained {model_name} model")
        if model_name == 'yolov8':
            return YOLO('yolov8n.pt')  # or yolov8s.pt, yolov8m.pt, etc.
        elif model_name == 'yolov11':
            return YOLO('yolo11n.pt')  # or yolo11s.pt, yolo11m.pt, etc.
        elif model_name == 'yolov12':
            return YOLO('yolo12n.pt')  # or yolo12s.pt, yolo12m.pt, etc.
        else:
            raise ValueError(f"Unknown model: {model_name}")

def get_all_images(data_dir):
    """Get all image paths from directory"""
    image_paths = []
    if os.path.isdir(data_dir):
        for img_file in sorted(os.listdir(data_dir)):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                image_paths.append(os.path.join(data_dir, img_file))
    return image_paths

def extract_detections(results):
    """
    Extract detections from YOLO results
    
    Returns:
        List of detections, each as [x1, y1, x2, y2, conf, class]
    """
    detections = []
    if results and len(results) > 0:
        result = results[0]
        if result.boxes is not None:
            boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
            confidences = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)
            
            for i in range(len(boxes)):
                detections.append([
                    boxes[i][0], boxes[i][1], boxes[i][2], boxes[i][3],  # x1, y1, x2, y2
                    confidences[i],  # confidence
                    classes[i]  # class
                ])
    return np.array(detections) if detections else np.array([]).reshape(0, 6)

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes
    Box format: [x1, y1, x2, y2]
    """
    x1_min, y1_min, x1_max, y1_max = box1[:4]
    x2_min, y2_min, x2_max, y2_max = box2[:4]
    
    # Calculate intersection
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    if inter_x_max < inter_x_min or inter_y_max < inter_y_min:
        return 0.0
    
    inter_area = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
    
    # Calculate union
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0

def weighted_nms(detections_list, weights=None, iou_threshold=0.5, conf_threshold=0.25):
    """
    Weighted Non-Maximum Suppression for ensemble detections
    
    Args:
        detections_list: List of detection arrays from different models
        weights: Weights for each model (default: equal weights)
        iou_threshold: IoU threshold for NMS
        conf_threshold: Confidence threshold
    
    Returns:
        Final detections after weighted NMS
    """
    if weights is None:
        weights = [1.0 / len(detections_list)] * len(detections_list)
    
    # Combine all detections with model weights
    all_detections = []
    for detections, weight in zip(detections_list, weights):
        if len(detections) > 0:
            # Weight the confidence scores
            weighted_detections = detections.copy()
            weighted_detections[:, 4] *= weight  # Multiply confidence by weight
            all_detections.append(weighted_detections)
    
    if not all_detections:
        return np.array([]).reshape(0, 6)
    
    # Concatenate all detections
    combined = np.vstack(all_detections)
    
    # Filter by confidence
    combined = combined[combined[:, 4] >= conf_threshold]
    
    if len(combined) == 0:
        return np.array([]).reshape(0, 6)
    
    # Sort by confidence (descending)
    sorted_indices = np.argsort(-combined[:, 4])
    combined = combined[sorted_indices]
    
    # Apply NMS
    keep = []
    while len(combined) > 0:
        # Take the highest confidence detection
        keep.append(combined[0])
        
        if len(combined) == 1:
            break
        
        # Calculate IoU with remaining boxes
        ious = []
        for i in range(1, len(combined)):
            iou = calculate_iou(combined[0], combined[i])
            ious.append(iou)
        
        ious = np.array(ious)
        
        # Keep boxes with IoU < threshold (and same class)
        same_class = combined[1:, 5] == combined[0, 5]
        low_iou = ious < iou_threshold
        mask = same_class & low_iou
        
        combined = combined[1:][mask]
    
    return np.array(keep) if keep else np.array([]).reshape(0, 6)


# 4. Prepare Data and Load Models


In [ ]:
# Get all image paths
train_images = get_all_images(TRAIN_DIR)
val_images = get_all_images(VAL_DIR)

print(f"Training images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")

# Load YOLO models
BASE_MODELS = ['yolov8', 'yolov11', 'yolov12']
yolo_models = {}

for model_name in BASE_MODELS:
    model_path = MODEL_PATHS.get(model_name)
    try:
        yolo_models[model_name] = load_yolo_model(model_name, model_path)
        print(f"✓ {model_name.upper()} loaded successfully")
    except Exception as e:
        print(f"✗ Failed to load {model_name}: {e}")
        # Remove from base models if loading failed
        if model_name in BASE_MODELS:
            BASE_MODELS.remove(model_name)

print(f"\nLoaded {len(yolo_models)} models: {list(yolo_models.keys())}")


# 5. Generate Base Model Predictions with Cross-Validation


In [ ]:
def predict_with_model(model, image_paths, conf_threshold=CONF_THRESHOLD):
    """
    Get predictions from a YOLO model for all images
    
    Returns:
        Dictionary mapping image_path -> detections array
    """
    predictions = {}
    for img_path in image_paths:
        try:
            results = model.predict(
                img_path,
                conf=conf_threshold,
                iou=IOU_THRESHOLD,
                imgsz=IMG_SIZE,
                verbose=False
            )
            detections = extract_detections(results)
            predictions[img_path] = detections
        except Exception as e:
            print(f"Error predicting {img_path}: {e}")
            predictions[img_path] = np.array([]).reshape(0, 6)
    return predictions

# Initialize storage for predictions
train_oof_predictions = {model: {} for model in BASE_MODELS}
val_predictions = {model: {} for model in BASE_MODELS}

# K-Fold for cross-validation
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

print("Generating base model predictions with cross-validation...")
print("=" * 60)

for model_name in BASE_MODELS:
    print(f"\nProcessing {model_name.upper()}...")
    model = yolo_models[model_name]
    
    # Cross-validation loop
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_images)):
        print(f"  Fold {fold + 1}/{N_SPLITS}...", end=' ')
        
        # Get validation images for this fold
        val_fold_images = [train_images[i] for i in val_idx]
        val_fold_preds = predict_with_model(model, val_fold_images)
        
        # Store OOF predictions
        for img_path, detections in val_fold_preds.items():
            train_oof_predictions[model_name][img_path] = detections
        
        print("Done")
    
    # Get predictions on full validation set
    print("  Generating validation predictions...", end=' ')
    val_preds = predict_with_model(model, val_images)
    val_predictions[model_name] = val_preds
    print("Done")
    
    print(f"  {model_name.upper()} completed!")

print("\n" + "=" * 60)
print("All base model predictions generated!")


# 6. Prepare Features for Stacking


In [ ]:
def extract_features_for_stacking(predictions_dict, image_paths):
    """
    Extract features from detections for stacking
    
    For each image, extract:
    - Number of detections per model
    - Average confidence per model
    - Max confidence per model
    - Box coordinates (normalized) for top N detections
    - Class predictions for top N detections
    """
    max_detections = 10  # Maximum number of detections to consider per image
    
    features_list = []
    
    for img_path in image_paths:
        features = []
        
        for model_name in BASE_MODELS:
            detections = predictions_dict[model_name].get(img_path, np.array([]).reshape(0, 6))
            
            if len(detections) > 0:
                # Sort by confidence
                sorted_idx = np.argsort(-detections[:, 4])
                detections = detections[sorted_idx]
                
                # Take top N
                n_det = min(len(detections), max_detections)
                detections = detections[:n_det]
                
                # Extract features
                num_detections = len(detections)
                avg_conf = np.mean(detections[:, 4]) if num_detections > 0 else 0.0
                max_conf = np.max(detections[:, 4]) if num_detections > 0 else 0.0
                
                # Box coordinates (normalized to 0-1)
                boxes = detections[:, :4].flatten()  # Flatten all box coordinates
                confidences = detections[:, 4]
                classes = detections[:, 5].astype(int)
                
                # Pad to max_detections
                if num_detections < max_detections:
                    pad_size = max_detections - num_detections
                    boxes = np.pad(boxes, (0, pad_size * 4), mode='constant')
                    confidences = np.pad(confidences, (0, pad_size), mode='constant')
                    classes = np.pad(classes, (0, pad_size), mode='constant', constant_values=-1)
                
                # Combine features
                model_features = np.concatenate([
                    [num_detections, avg_conf, max_conf],
                    boxes[:max_detections * 4],
                    confidences[:max_detections],
                    classes[:max_detections]
                ])
            else:
                # No detections
                model_features = np.zeros(3 + max_detections * 4 + max_detections + max_detections)
                model_features[2] = -1  # Mark max_conf as -1 to indicate no detections
            
            features.extend(model_features)
        
        features_list.append(features)
    
    return np.array(features_list)

# Extract features for stacking
print("Extracting features for stacking...")
X_train_stack = extract_features_for_stacking(train_oof_predictions, train_images)
X_val_stack = extract_features_for_stacking(val_predictions, val_images)

print(f"Training stacking features shape: {X_train_stack.shape}")
print(f"Validation stacking features shape: {X_val_stack.shape}")


# 7. Method 1: Weighted NMS Ensemble (Direct Detection Fusion)


In [ ]:
# Weighted NMS - Simple and effective for detection
# This method directly combines detections from all models

def evaluate_detections(predictions_dict, image_paths, ground_truth=None):
    """
    Evaluate detection predictions
    For now, we'll use average confidence and number of detections as metrics
    """
    total_detections = 0
    total_conf = 0.0
    num_images = len(image_paths)
    
    for img_path in image_paths:
        detections = predictions_dict.get(img_path, np.array([]).reshape(0, 6))
        total_detections += len(detections)
        if len(detections) > 0:
            total_conf += np.mean(detections[:, 4])
    
    avg_detections = total_detections / num_images if num_images > 0 else 0
    avg_conf = total_conf / num_images if num_images > 0 else 0
    
    return avg_detections, avg_conf

# Test different weight combinations
print("Testing Weighted NMS with different weights...")
print("=" * 60)

# Equal weights
equal_weights = [1.0 / len(BASE_MODELS)] * len(BASE_MODELS)
print(f"\nEqual weights: {dict(zip(BASE_MODELS, equal_weights))}")

# Generate ensemble predictions with equal weights
val_ensemble_equal = {}
for img_path in val_images:
    detections_list = [val_predictions[model].get(img_path, np.array([]).reshape(0, 6)) 
                       for model in BASE_MODELS]
    ensemble_detections = weighted_nms(detections_list, equal_weights, IOU_THRESHOLD, CONF_THRESHOLD)
    val_ensemble_equal[img_path] = ensemble_detections

avg_det, avg_conf = evaluate_detections(val_ensemble_equal, val_images)
print(f"Average detections per image: {avg_det:.2f}")
print(f"Average confidence: {avg_conf:.4f}")

# Optimize weights based on validation performance
from scipy.optimize import minimize

def objective_weights(weights):
    """Objective: maximize average confidence while maintaining reasonable detection count"""
    weights_dict = {BASE_MODELS[i]: weights[i] for i in range(len(BASE_MODELS))}
    
    # Generate ensemble predictions
    ensemble_preds = {}
    for img_path in val_images:
        detections_list = [val_predictions[model].get(img_path, np.array([]).reshape(0, 6)) 
                          for model in BASE_MODELS]
        ensemble_detections = weighted_nms(detections_list, weights, IOU_THRESHOLD, CONF_THRESHOLD)
        ensemble_preds[img_path] = ensemble_detections
    
    avg_det, avg_conf = evaluate_detections(ensemble_preds, val_images)
    
    # Objective: maximize confidence, penalize too few or too many detections
    # Target: 2-5 detections per image (adjust based on your dataset)
    target_detections = 3.0
    det_penalty = abs(avg_det - target_detections) * 0.1
    
    return -(avg_conf - det_penalty)  # Minimize negative (maximize positive)

# Optimize weights
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
bounds = [(0, 1) for _ in BASE_MODELS]
initial_weights = [1.0 / len(BASE_MODELS)] * len(BASE_MODELS)

result = minimize(objective_weights, initial_weights, method='SLSQP', 
                  bounds=bounds, constraints=constraints)

optimized_weights = {BASE_MODELS[i]: result.x[i] for i in range(len(BASE_MODELS))}
print(f"\nOptimized weights: {optimized_weights}")

# Generate final ensemble predictions
val_ensemble_optimized = {}
for img_path in val_images:
    detections_list = [val_predictions[model].get(img_path, np.array([]).reshape(0, 6)) 
                       for model in BASE_MODELS]
    ensemble_detections = weighted_nms(detections_list, list(optimized_weights.values()), 
                                       IOU_THRESHOLD, CONF_THRESHOLD)
    val_ensemble_optimized[img_path] = ensemble_detections

avg_det_opt, avg_conf_opt = evaluate_detections(val_ensemble_optimized, val_images)
print(f"\nOptimized ensemble:")
print(f"Average detections per image: {avg_det_opt:.2f}")
print(f"Average confidence: {avg_conf_opt:.4f}")

# Store best weights
BEST_WEIGHTS = list(optimized_weights.values())


In [ ]:
# For detection, we can use meta-learners to:
# 1. Predict if a detection should be kept (binary classification)
# 2. Refine bounding box coordinates (regression)
# 3. Predict class probabilities (multi-class classification)

# This is a simplified approach - in practice, you'd need ground truth annotations
# For now, we'll use the weighted NMS results as "pseudo-labels"

print("Note: Meta-learner approach requires ground truth annotations.")
print("Using weighted NMS as the primary ensemble method.")
print("\nFor full meta-learner implementation, you would:")
print("1. Extract features from each detection (box coordinates, confidence, class)")
print("2. Train a classifier to predict if detection is correct (requires GT)")
print("3. Train a regressor to refine box coordinates (requires GT)")
print("4. Train a classifier to refine class predictions (requires GT)")


# 9. Visualize Ensemble Results


In [ ]:
def visualize_detections(image_path, detections, title="Detections", save_path=None):
    """Visualize detections on an image"""
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image: {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img_rgb)
    
    if len(detections) > 0:
        for det in detections:
            x1, y1, x2, y2, conf, cls = det
            width = x2 - x1
            height = y2 - y1
            
            # Draw bounding box
            rect = patches.Rectangle(
                (x1, y1), width, height,
                linewidth=2, edgecolor='red', facecolor='none'
            )
            ax.add_patch(rect)
            
            # Add label
            label = f"Class {int(cls)}: {conf:.2f}"
            ax.text(x1, y1 - 5, label, color='red', fontsize=10, weight='bold')
    
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()

# Visualize individual models vs ensemble
if len(val_images) > 0:
    sample_image = val_images[0]
    
    print("Individual Model Predictions:")
    print("=" * 60)
    
    for model_name in BASE_MODELS:
        detections = val_predictions[model_name].get(sample_image, np.array([]).reshape(0, 6))
        print(f"\n{model_name.upper()}: {len(detections)} detections")
        if len(detections) > 0:
            print(f"  Average confidence: {np.mean(detections[:, 4]):.4f}")
            visualize_detections(sample_image, detections, 
                               title=f"{model_name.upper()} Detections")
    
    print("\nEnsemble (Weighted NMS) Predictions:")
    print("=" * 60)
    ensemble_detections = val_ensemble_optimized.get(sample_image, np.array([]).reshape(0, 6))
    print(f"\nEnsemble: {len(ensemble_detections)} detections")
    if len(ensemble_detections) > 0:
        print(f"  Average confidence: {np.mean(ensemble_detections[:, 4]):.4f}")
    visualize_detections(sample_image, ensemble_detections, 
                        title="Ensemble (Weighted NMS) Detections")


# 10. Generate Predictions for Test Set (Kaggle Submission)


In [ ]:
def predict_test_set_ensemble(test_dir, yolo_models_dict, weights):
    """
    Generate ensemble predictions for test set
    
    Args:
        test_dir: Directory containing test images
        yolo_models_dict: Dictionary of YOLO models {model_name: model}
        weights: List of weights for each model
    
    Returns:
        Dictionary mapping image_path -> detections array
    """
    test_images = get_all_images(test_dir)
    print(f"Found {len(test_images)} test images")
    
    # Get predictions from all models
    test_predictions = {}
    for model_name, model in yolo_models_dict.items():
        print(f"Generating {model_name} predictions...", end=' ')
        preds = predict_with_model(model, test_images)
        test_predictions[model_name] = preds
        print("Done")
    
    # Apply weighted NMS
    print("Applying weighted NMS ensemble...", end=' ')
    ensemble_predictions = {}
    for img_path in test_images:
        detections_list = [test_predictions[model].get(img_path, np.array([]).reshape(0, 6)) 
                          for model in BASE_MODELS]
        ensemble_detections = weighted_nms(detections_list, weights, IOU_THRESHOLD, CONF_THRESHOLD)
        ensemble_predictions[img_path] = ensemble_detections
    print("Done")
    
    return ensemble_predictions

# Example usage (uncomment when test directory is available)
# TEST_DIR = '/kaggle/input/car-damage-detection/test'  # Adjust path
# 
# test_ensemble_predictions = predict_test_set_ensemble(TEST_DIR, yolo_models, BEST_WEIGHTS)
# 
# # Convert to COCO format for submission (if required)
# def detections_to_coco_format(predictions_dict, image_paths, output_file):
#     """Convert detections to COCO format JSON"""
#     coco_output = {
#         "images": [],
#         "annotations": [],
#         "categories": []  # Add your category definitions
#     }
#     
#     ann_id = 0
#     for img_id, img_path in enumerate(image_paths):
#         img = cv2.imread(img_path)
#         h, w = img.shape[:2]
#         
#         # Add image info
#         coco_output["images"].append({
#             "id": img_id,
#             "file_name": os.path.basename(img_path),
#             "width": w,
#             "height": h
#         })
#         
#         # Add detections
#         detections = predictions_dict.get(img_path, np.array([]).reshape(0, 6))
#         for det in detections:
#             x1, y1, x2, y2, conf, cls = det
#             bbox = [x1, y1, x2 - x1, y2 - y1]  # COCO format: [x, y, width, height]
#             
#             coco_output["annotations"].append({
#                 "id": ann_id,
#                 "image_id": img_id,
#                 "category_id": int(cls),
#                 "bbox": bbox,
#                 "score": float(conf),
#                 "area": (x2 - x1) * (y2 - y1)
#             })
#             ann_id += 1
#     
#     with open(output_file, 'w') as f:
#         json.dump(coco_output, f)
#     
#     print(f"Saved COCO format predictions to {output_file}")
# 
# # Save predictions
# output_file = 'submission.json'
# detections_to_coco_format(test_ensemble_predictions, test_images, output_file)


# 11. Compare Individual Models vs Ensemble


In [ ]:
# Compare performance metrics
print("=" * 60)
print("Model Comparison:")
print("=" * 60)

comparison_data = []

for model_name in BASE_MODELS:
    avg_det, avg_conf = evaluate_detections(val_predictions[model_name], val_images)
    comparison_data.append({
        'Model': model_name.upper(),
        'Avg Detections': avg_det,
        'Avg Confidence': avg_conf
    })
    print(f"{model_name.upper():15s}: {avg_det:6.2f} detections, {avg_conf:.4f} avg confidence")

# Ensemble results
avg_det_ens, avg_conf_ens = evaluate_detections(val_ensemble_optimized, val_images)
comparison_data.append({
    'Model': 'ENSEMBLE (Weighted NMS)',
    'Avg Detections': avg_det_ens,
    'Avg Confidence': avg_conf_ens
})
print(f"{'ENSEMBLE':15s}: {avg_det_ens:6.2f} detections, {avg_conf_ens:.4f} avg confidence")
print("=" * 60)

# Visualization
df_comparison = pd.DataFrame(comparison_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Average detections
ax1.bar(df_comparison['Model'], df_comparison['Avg Detections'], color='skyblue')
ax1.set_title('Average Detections per Image')
ax1.set_ylabel('Number of Detections')
ax1.tick_params(axis='x', rotation=45)

# Average confidence
ax2.bar(df_comparison['Model'], df_comparison['Avg Confidence'], color='lightcoral')
ax2.set_title('Average Confidence Score')
ax2.set_ylabel('Confidence')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# 12. Save Models and Predictions


In [ ]:
import joblib
import pickle

# Save optimized weights
weights_dict = {BASE_MODELS[i]: BEST_WEIGHTS[i] for i in range(len(BASE_MODELS))}
with open('ensemble_weights.pkl', 'wb') as f:
    pickle.dump(weights_dict, f)
print("Saved ensemble weights as 'ensemble_weights.pkl'")

# Save predictions (optional, can be large)
# np.save('val_predictions.npy', val_predictions)
# print("Saved validation predictions")

# Save configuration
config = {
    'models': BASE_MODELS,
    'weights': weights_dict,
    'conf_threshold': CONF_THRESHOLD,
    'iou_threshold': IOU_THRESHOLD,
    'img_size': IMG_SIZE
}

with open('ensemble_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("Saved ensemble configuration as 'ensemble_config.json'")


# Summary and Notes
# ===========================================
# 
# **Key Points:**
# 1. This notebook implements ensemble stacking for YOLO object detection models
# 2. Uses Weighted Non-Maximum Suppression (NMS) to combine detections from multiple models
# 3. Optimizes model weights based on validation performance
# 4. Supports YOLOv8, YOLOv11, and YOLOv12 models
# 
# **Ensemble Methods:**
# - **Weighted NMS**: Primary method - combines detections with optimized weights
# - **Meta-Learner**: Advanced approach (requires ground truth annotations)
# 
# **For Kaggle Submission:**
# - Adjust paths in Section 2 to match your Kaggle dataset structure
# - Uncomment and modify Section 10 to generate test predictions
# - Ensure all YOLO model weights are available
# - Convert predictions to required format (COCO, YOLO, etc.)
# 
# **Tips:**
# - Weighted NMS is more effective than simple averaging for object detection
# - Adjust IOU_THRESHOLD and CONF_THRESHOLD based on your dataset
# - Consider using different confidence thresholds for different models
# - For better performance, train models with different architectures or data augmentations
# - Evaluate using mAP (mean Average Precision) if ground truth is available
# 
# **Advanced Improvements:**
# - Implement class-specific NMS
# - Use different IOU thresholds for different classes
# - Implement temporal consistency for video sequences
# - Add confidence calibration
# - Use test-time augmentation (TTA) for each model before ensemble
